# HW1 - Kinematics and Dynamics of Rotation

## 1) Helper functions (visualize.py)

- Hamilton product
- Quaternion derivative from angular velocity
- Exact incremental quaternion via exponential map
- Quaternion to rotation matrix conversion

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation


def visualize_rotation(rotation_matrices,
                       stats=None,
                       verbose=True,
                       show=True):
    # Track body axes through time:
    # axis i trajectory is rotation_matrices[:, i, k] for k=0..N-1
    axes_t = np.array([rotation_matrices[:, 0, :],
                       rotation_matrices[:, 1, :],
                       rotation_matrices[:, 2, :]])

    N = rotation_matrices.shape[2]
    frames = N
    # FuncAnimation expects interval in milliseconds.
    interval = max(20, int(1000 / max(N, 1)))

    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    ax.axis('off')

    colors = ['r', 'g', 'b']

    lines = sum([ax.plot([], [], [], '--', c=c, alpha=0.3)
                for c in colors], [])
    axes = sum([ax.plot([], [], [], '-', c=c, lw=3)
                for c in colors], [])
    pts = sum([ax.plot([], [], [], 'o', c=c, lw=5)
               for c in colors], [])

    corner_text = ax.text(0.02, 0.90, 1.1, '')

    ax.set_xlim((-1.1, 1.1))
    ax.set_ylim((-1.1, 1.1))
    ax.set_zlim((-1.1, 1.1))

    ax.view_init(14, 0)
    lag = 35

    def init():
        for line, pt, axs, xi in zip(lines, pts, axes, axes_t):
            x, y, z = xi[:, :1]
            line.set_data(x[-lag:], y[-lag:])
            line.set_3d_properties(z[-lag:])
            axs.set_data(np.hstack((0, x[-1:])), np.hstack((0, y[-1:])))
            axs.set_3d_properties(np.hstack((0, z[-1:])))

            pt.set_data(x[-1:], y[-1:])
            pt.set_3d_properties(z[-1:])

        return lines + pts + axes

    def animate(i):
        j = i + 1
        stat_text = ''

        for line, pt, axs, xi in zip(lines, pts, axes, axes_t):
            x, y, z = xi[:, :j]
            line.set_data(x[-lag:], y[-lag:])
            line.set_3d_properties(z[-lag:])
            axs.set_data(np.hstack((0, x[-1:])), np.hstack((0, y[-1:])))
            axs.set_3d_properties(np.hstack((0, z[-1:])))

            pt.set_data(x[-1:], y[-1:])
            pt.set_3d_properties(z[-1:])

        if stats is not None:
            idx = min(j - 1, len(stats[0][1]) - 1)
            for stat in stats:
                stat_text += rf'{stat[0]}:  {round(stat[1][idx], 3)}'
                stat_text += '\n'
            corner_text.set_text(stat_text)

        fig.canvas.draw()
        return lines + pts + axes


    anim = animation.FuncAnimation(
        fig,
        animate,
        init_func=init,
        frames=frames,
        interval=interval,
        blit=False
    )

    if show:
        plt.show()

    return anim


def quat_multiply(q, p):
    """Hamilton product q ⊗ p for quaternions [w, x, y, z]."""
    w1, x1, y1, z1 = q
    w2, x2, y2, z2 = p
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], dtype=float)


def quat_normalize(q):
    """Normalize a quaternion to unit length."""
    norm = np.linalg.norm(q)
    if norm < 1e-12:
        return q.copy()
    return q / norm


def quat_derivative(q, omega):
    """q_dot = 1/2 * q ⊗ [0, omega]."""
    omega_quat = np.array([0.0, omega[0], omega[1], omega[2]], dtype=float)
    return 0.5 * quat_multiply(q, omega_quat)


def delta_quat_exact(omega, dt, eps=1e-12):
    """
    Exact increment from exp((1/2)*omega_hat*dt).
    For omega != 0:
      dq = [cos(theta/2), u*sin(theta/2)]
      where theta = ||omega||*dt and u = omega/||omega||
    """
    w_norm = np.linalg.norm(omega)
    if w_norm < eps:
        return np.array([1.0, 0.0, 0.0, 0.0], dtype=float)

    theta = w_norm * dt
    u = omega / w_norm
    return np.hstack((np.cos(theta / 2.0), u * np.sin(theta / 2.0)))


def quat_to_rotmat(q):
    """Convert unit quaternion [w, x, y, z] to 3x3 rotation matrix."""
    # Normalize defensively to avoid numerical issues when converting to R
    q = q / np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)]
    ], dtype=float)

In [ ]:
def quaternion_kinematics_step(q, omega, dt):
    """Lie-group update for quaternion kinematics."""
    dq = delta_quat_exact(omega, dt)
    q_next = quat_multiply(q, dq)
    return quat_normalize(q_next)

## 2) Simulation setup

- Initial quaternion $q_0 = [1,0,0,0]$ (identity rotation)
- Constant angular velocity $\omega$
- Uniform time step $dt$

In [ ]:
# Time grid
T = 12.0
dt = 0.02
N = int(T / dt)
t = np.arange(N + 1) * dt

# Angular velocity in body frame [rad/s]
omega = np.array([0.7, 1.1, 0.5], dtype=float)

# Initial orientation quaternion
q0 = np.array([1.0, 0.0, 0.0, 0.0], dtype=float)

print(f'Total steps: {N}')
print(f'dt: {dt}')
print(f'omega: {omega}')

## 3) Integrate quaternion kinematics

Subtask 1 and Subtask 5:
- **Exact update**: $q_{k+1}=q_k\otimes\exp(\frac{1}{2}\hat{\omega}dt)$
- **Forward Euler**: $q_{k+1}=q_k + dt\,\dot q_k$, with $\dot q = \frac{1}{2} q\otimes[0,\omega]$

In [ ]:
# Storage for both methods
q_exact = np.zeros((N + 1, 4), dtype=float)
q_euler = np.zeros((N + 1, 4), dtype=float)

q_exact[0] = q0
q_euler[0] = q0

# Precompute exact increment (constant omega case)
dq = delta_quat_exact(omega, dt)

for k in range(N):
    # Exact Lie-group update: stays very close to unit norm numerically
    q_exact[k + 1] = quat_multiply(q_exact[k], dq)

    # Forward Euler update: simple but introduces norm drift
    q_euler[k + 1] = q_euler[k] + dt * quat_derivative(q_euler[k], omega)

## 4) Check quaternion norms (Subtask 2 + compare)

In [ ]:
norm_exact = np.linalg.norm(q_exact, axis=1)
norm_euler = np.linalg.norm(q_euler, axis=1)

print(f'Exact method norm range:  [{norm_exact.min():.8f}, {norm_exact.max():.8f}]')
print(f'Euler method norm range:  [{norm_euler.min():.8f}, {norm_euler.max():.8f}]')

plt.figure(figsize=(9, 4))
plt.plot(t, norm_exact, label='Exact Lie integration', linewidth=2)
plt.plot(t, norm_euler, label='Forward Euler', linewidth=2)
plt.axhline(1.0, color='k', linestyle='--', alpha=0.6, label='Unit norm')
plt.title('Quaternion norm comparison')
plt.xlabel('Time [s]')
plt.ylabel('||q||')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 5) Plot quaternion trajectories $q_k$ (Subtask 3 + compare)

In [ ]:
labels = ['q_w', 'q_x', 'q_y', 'q_z']

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
for i in range(4):
    axes[i].plot(t, q_exact[:, i], label='Exact', linewidth=2)
    axes[i].plot(t, q_euler[:, i], label='Euler', linewidth=1.6, linestyle='--')
    axes[i].set_ylabel(labels[i])
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('Quaternion trajectory components')
axes[-1].set_xlabel('Time [s]')
axes[0].legend(loc='upper right')
plt.tight_layout()
plt.show()

## 6) Export animations (Subtask 4 + compare)

- Files are saved in `HW1/outputs/`

In [ ]:
# Convert quaternion sequences to rotation matrix sequences
R_exact = np.stack([quat_to_rotmat(q) for q in q_exact], axis=2)
R_euler = np.stack([quat_to_rotmat(q) for q in q_euler], axis=2)

# Build animations
anim_exact = visualize_rotation(
    R_exact,
    stats=[('||q||', norm_exact)],
    verbose=True,
    show=False
)

anim_euler = visualize_rotation(
    R_euler,
    stats=[('||q||', norm_euler)],
    verbose=True,
    show=False
)

In [ ]:
from pathlib import Path
from matplotlib.animation import FFMpegWriter, PillowWriter

out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)

def export_animation(anim, base_name, out_dir, fps_mp4=30, fps_gif=20):
    mp4_path = out_dir / f'{base_name}.mp4'
    gif_path = out_dir / f'{base_name}.gif'

    try:
        anim.save(mp4_path, writer=FFMpegWriter(fps=fps_mp4), dpi=120)
        print(f'Saved: {mp4_path.resolve()}')
        return mp4_path
    except Exception as err_mp4:
        print(f'MP4 export failed for {base_name}: {err_mp4}')
        print('Trying GIF export with PillowWriter...')
        try:
            anim.save(gif_path, writer=PillowWriter(fps=fps_gif), dpi=100)
            print(f'Saved: {gif_path.resolve()}')
            return gif_path
        except Exception as err_gif:
            print(f'GIF export failed for {base_name}: {err_gif}')
            return None

saved_exact = export_animation(anim_exact, 'rotation_exact', out_dir)
saved_euler = export_animation(anim_euler, 'rotation_euler', out_dir)

plt.close('all')

saved_exact, saved_euler

## 7) Problem 2 - Newton-Euler dynamics integration


### 7.1) Model setup

Constant mass, diagonal inertia, gravity in the world frame, and constant body-frame force/torque inputs.
The state is $x = [q, \omega, p, v]$ with the quaternion stored first so the same Lie-group update can be reused.

In [ ]:
# Time discretization for the rigid-body simulation
tf2 = 12.0
dt2 = 0.01
t2 = np.arange(0.0, tf2 + dt2, dt2)

# Physical parameters
mass2 = 1.8
inertia2 = np.diag([2.0, 1.3, 0.8])
gravity2 = np.array([0.0, 0.0, -9.81], dtype=float)

# Constant external inputs in the body frame
force_body2 = np.array([4.0, 0.0, 18.0], dtype=float)
torque_body2 = np.array([0.15, -0.05, 0.10], dtype=float)

# Initial conditions
q0_2 = np.array([1.0, 0.0, 0.0, 0.0], dtype=float)
omega0_2 = np.array([0.6, 1.0, 0.3], dtype=float)
position0_2 = np.array([0.0, 0.0, 0.0], dtype=float)
velocity0_2 = np.array([1.0, 0.4, 0.2], dtype=float)

# State array: [q(4), omega(3), position(3), velocity(3)]
state2 = np.zeros((len(t2), 13), dtype=float)
state2[0, :4] = quat_normalize(q0_2)
state2[0, 4:7] = omega0_2
state2[0, 7:10] = position0_2
state2[0, 10:13] = velocity0_2

# The quaternion part is advanced with the same Lie-group update from Problem 1.
for k in range(len(t2) - 1):
    q = state2[k, :4]
    omega = state2[k, 4:7]
    position = state2[k, 7:10]
    velocity = state2[k, 10:13]

    rotation = quat_to_rotmat(q)
    omega_dot = np.linalg.solve(inertia2, torque_body2 - np.cross(omega, inertia2 @ omega))
    omega_next = omega + dt2 * omega_dot
    omega_mid = 0.5 * (omega + omega_next)
    q_next = quaternion_kinematics_step(q, omega_mid, dt2)

    acceleration = gravity2 + (rotation @ force_body2) / mass2
    velocity_next = velocity + dt2 * acceleration
    position_next = position + dt2 * velocity

    state2[k + 1] = np.hstack((q_next, omega_next, position_next, velocity_next))

q_hist2 = state2[:, :4]
omega_hist2 = state2[:, 4:7]
position_hist2 = state2[:, 7:10]
velocity_hist2 = state2[:, 10:13]
rotation_hist2 = np.stack([quat_to_rotmat(q) for q in q_hist2], axis=0)
quat_norm2 = np.linalg.norm(q_hist2, axis=1)

print(f'Time steps: {len(t2) - 1}')
print(f'Quaternion norm range: [{quat_norm2.min():.8f}, {quat_norm2.max():.8f}]')

### 7.2) Plots


In [ ]:
# Position components
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
labels2 = ['p_x', 'p_y', 'p_z']
for i in range(3):
    axes[i].plot(t2, position_hist2[:, i], linewidth=2)
    axes[i].set_ylabel(labels2[i])
    axes[i].grid(True, alpha=0.3)
axes[0].set_title('Problem 2: Position components')
axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

# Velocity components
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
labels2 = ['v_x', 'v_y', 'v_z']
for i in range(3):
    axes[i].plot(t2, velocity_hist2[:, i], linewidth=2)
    axes[i].set_ylabel(labels2[i])
    axes[i].grid(True, alpha=0.3)
axes[0].set_title('Problem 2: Velocity components')
axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

# Quaternion components
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
labels2 = ['q_0', 'q_1', 'q_2', 'q_3']
for i in range(4):
    axes[i].plot(t2, q_hist2[:, i], linewidth=2)
    axes[i].set_ylabel(labels2[i])
    axes[i].grid(True, alpha=0.3)
axes[0].set_title('Problem 2: Quaternion components')
axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

# Angular velocity components
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
labels2 = ['omega_x', 'omega_y', 'omega_z']
for i in range(3):
    axes[i].plot(t2, omega_hist2[:, i], linewidth=2)
    axes[i].set_ylabel(labels2[i])
    axes[i].grid(True, alpha=0.3)
axes[0].set_title('Problem 2: Angular velocity components')
axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

### 7.3) Visualization and export


In [ ]:
def rigid_body_visualize(positions, rotation_matrices, stats=None, verbose=True, show=True, stride=5):
    """Animate the rigid body center of mass and body axes."""
    positions = positions[::stride]
    rotation_matrices = rotation_matrices[::stride]
    if stats is not None:
        stats = [(label, values[::stride]) for label, values in stats]

    frames = len(positions)
    interval = max(30, int(1000 / 20))

    mins = positions.min(axis=0)
    maxs = positions.max(axis=0)
    center = 0.5 * (mins + maxs)
    radius = max(0.5 * np.max(maxs - mins), 1.0)

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(projection='3d')
    ax.axis('off')
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)
    ax.view_init(18, 35)

    trail_line, = ax.plot([], [], [], 'k--', lw=1.5, alpha=0.4)
    colors = ['r', 'g', 'b']
    axis_lines = [ax.plot([], [], [], '-', c=c, lw=3)[0] for c in colors]
    axis_pts = [ax.plot([], [], [], 'o', c=c, ms=4)[0] for c in colors]
    center_pt, = ax.plot([], [], [], 'ko', ms=5)
    corner_text = ax.text2D(0.02, 0.94, '', transform=ax.transAxes)

    def init():
        trail_line.set_data([], [])
        trail_line.set_3d_properties([])
        for line, pt in zip(axis_lines, axis_pts):
            line.set_data([], [])
            line.set_3d_properties([])
            pt.set_data([], [])
            pt.set_3d_properties([])
        center_pt.set_data([], [])
        center_pt.set_3d_properties([])
        corner_text.set_text('')
        return [trail_line, center_pt, corner_text] + axis_lines + axis_pts

    def animate(i):
        j = i + 1
        p = positions[j - 1]
        R = rotation_matrices[j - 1]

        trail = positions[:j]
        trail_line.set_data(trail[:, 0], trail[:, 1])
        trail_line.set_3d_properties(trail[:, 2])

        for idx, (line, pt) in enumerate(zip(axis_lines, axis_pts)):
            direction = R[:, idx]
            tip = p + 0.25 * direction
            line.set_data([p[0], tip[0]], [p[1], tip[1]])
            line.set_3d_properties([p[2], tip[2]])
            pt.set_data([tip[0]], [tip[1]])
            pt.set_3d_properties([tip[2]])

        center_pt.set_data([p[0]], [p[1]])
        center_pt.set_3d_properties([p[2]])

        if stats is not None:
            stat_text = ''
            for label, values in stats:
                stat_text += f'{label}: {values[j - 1]: .4f}\n'
            corner_text.set_text(stat_text)

        return [trail_line, center_pt, corner_text] + axis_lines + axis_pts


    anim = animation.FuncAnimation(
        fig,
        animate,
        init_func=init,
        frames=frames,
        interval=interval,
        blit=False
    )

    if show:
        plt.show()

    return anim


anim2 = rigid_body_visualize(
    position_hist2,
    rotation_hist2,
    stats=[('||q||', quat_norm2)],
    verbose=True,
    show=False,
    stride=5
 )

saved_file2 = export_animation(anim2, 'rigid_body_motion_problem2', Path('outputs'))

from IPython.display import Video, display
if saved_file2 is not None:
    display(Video(str(saved_file2), embed=True))
saved_file2